# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
removed = df.duplicated().sum() #this counts how many duplicates are in the data set
print(removed) #can confirm we dropped the right number of rows by matching this number to the rows affected
df = df.drop_duplicates().copy() #df with the duplicates dropped, copied

log('duplicates','dropped duplicate rows', removed)

15
[duplicates] dropped duplicate rows (15 row(s))


**Markdown:** Deleted duplicates because we don't need to have multiple of the exact same data, this was likely an error and should be removed.

### TODO 2 — clean `price` -> float

In [4]:
# TODO
df['price'] = (df['price'].astype(str)
                                .str.replace('$','',regex = False)
                                .str.replace(',','', regex=False)
                                .str.strip()
                                .astype(float))
#we stripped away all the extra characters that could possible be in the price value and then converted it from a string to a float
assert df['price'].dtype == float #ensures that the price values have now been made floats and are no longer are strings
log('price','stripped away dollar signs, white space, and commas to change price to floats', len(df))

[price] stripped away dollar signs, white space, and commas to change price to floats (300 row(s))


**Markdown:** Changed price to float so that we can do numeric operations with it later as opposed to strings which we can't do math with.

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
revenue_before = (df['qty']*df['price']).sum() #For the final write up
print(f'Revenue Before:', revenue_before) #Looks at revenue before we make a big change in data
df['qty'] = pd.to_numeric(df['qty'], errors = 'coerce') #turns anything that can't be analyzed into NaN
missing = df['qty'].isna().sum() #sums the number of quantities that are missing
negative = (df['qty'] < 0).sum() #sums the number of quantities that are negative, or less than 0

df = df[df['qty'].notna() & (df['qty']>0)].copy() #makes the clean table only include quantities that are not missing and greater than 0, so not negative

revenue_after = (df['qty']*df['price']).sum() #revenue after deleting negative and missing quanity rows
print(f'Revenue After:', revenue_after)

print(f'missing:', missing) #printing number of quantities that have missing values
print(f'negative:', negative) #printing number of quantities that have negative values
log('missing', 'changed quantity to numeric and dropped missing rows', missing)
log('negative', 'changed quantity to numeric and dropped negative rows', negative )

Revenue Before: 4594.5
Revenue After: 4740.0
missing: 12
negative: 13
[missing] changed quantity to numeric and dropped missing rows (12 row(s))
[negative] changed quantity to numeric and dropped negative rows (13 row(s))


**Markdown:** Deleted missing values because making up new values to fill the holes would invalidate the data. Deleting negative values so that we can see overall revenue that usually would have been gained, also because maybe the negatives were an error.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
} #listed out all of the item names and what we want them to be instead
df['item'] = df['item'].replace(ITEM_MAP) #replaced item names with what we put in the item map
log('item_name','normalized item names so that punctuation differences would not impact the item name', len(df) )

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item_name] normalized item names so that punctuation differences would not impact the item name (275 row(s))


**Markdown:** Making sure all the item names align will help if we ever want to do analysis by item. Any mispelling or different punctuation can cause the code to think the same item are actually two different items which is not what we want.

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())
CATEGORY_MAP = {
    'Merch': 'Merch',
    'Food': 'Food',
    'food': 'Food',
    'rain-gear': 'RainGear',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
} #showing how we want the categories to be changed and represented -- here we decide that apparel should be counted in the merch category
df['category'] = df['category'].replace(CATEGORY_MAP) #replaces current categories with the category map we made
log('category_change','made sure category names matched no matter punctuation and added apparel to the merch category',len(df))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category_change] made sure category names matched no matter punctuation and added apparel to the merch category (275 row(s))


**Markdown:** I decided to combine apparel into merch becuase it makes more since for analyzing the data and buying decisions. For example if we want to choose what to restock, it is much easier to combine apparel into merch to better compare revenue across diverse categories.

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all() #asks if all items in df are one of these 3 options
# TODO: assert something about category
assert df['category'].isin(['Merch','Food','RainGear']).all() #asks if all categories across df are one of these 3 categories
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty']*df['price'] #creates a new revenue column by multiplying the existing prices and quantities
revenue_by_category = (df.groupby('category')['revenue'].sum().sort_values(ascending = False).round(2))
#the above line groups df by both category and revenue total then sorts by revenue descending (highest to lowest), then rounds answers to two decimals

revenue_by_category.loc['Overall Total'] = df['revenue'].sum().round(2)
#adding a total row at the bottom and rounding to two decimal places
revenue_by_category

,revenue
category,
Food,1656.0
Merch,1572.0
RainGear,1512.0
Overall Total,4740.0


**What I would tell the vendor:** I would tell a vendor to stock more food since that is the category with the highest revenue by almost a hundred dollars more than merch.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicate rows,15
1,price,"stripped away dollar signs, white space, and c...",300
2,missing,changed quantity to numeric and dropped missin...,12
3,negative,changed quantity to numeric and dropped negati...,13
4,item_name,normalized item names so that punctuation diff...,275
5,category_change,made sure category names matched no matter pun...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**Answer A:** TODO 3 where we dropped missing and negative quantities from the data set changed the revenue the most. Revenue before dropping missing and negatives quantities: 4594.50,
Revenue After: 4740.00

**Answer B:** We ended up dropping negative quantities where a normal person would have likely kept them. This was done because the lab didn't want any abnormal quantity values that weren't the normal positive ones, but because of this the revenue is skewed higher than it should be. The other choice of keeping the negative quantities would have shown up as a return to the business which would lower the revenue since those sales would no longer be counted.